# CohortX Task 2 — V4 tail completion over scored 0.81 CSV

**Kaggle settings:** Accelerator = None. Attach `Task_2.xlsx` through the
competition data and attach a Kaggle Dataset containing the Qwen3.5-9B GGUF.
Attach the exact `submission.csv` that scored 0.81 after renaming it to
`submission_081.csv`. This notebook preserves rows 0-44 without postprocess,
disables review, and replaces only skeleton fallback rows 45-49 with the LLM.
Internet is needed only if `llama-cpp-python>=0.3.30` is not already installed
and no compatible wheel has been attached. Inference itself makes no network
calls and uses `n_gpu_layers=0`.

**Metric**: adapted FM3S WordNet-based semantic similarity with Hungarian
matching between predicted and ground-truth triples (see competition Overview).

**Submission**: a CSV with the same two columns as the train sheet
(`eligibility_criteria`, `structured`), in the same row order as the test sheet.

In [1]:
from __future__ import annotations
import csv, importlib, json, os, re, subprocess, sys, time, hashlib
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import openpyxl

PROCESS_START = time.time()

# --- Path discovery: works locally and on Kaggle ---
# - Local Mac (default):   /Users/nnhzzz/Downloads/cohort-x-task-2/
# - Kaggle notebook:       data at /kaggle/input/competitions/cohort-x-task-2/,
#                          writable output at /kaggle/working/
# Override anything with COHORTX_XLSX / COHORTX_MODEL / COHORTX_OUT env vars.
def _discover_xlsx() -> Path:
    if (env := os.environ.get("COHORTX_XLSX")):
        return Path(env)
    candidates = [
        Path("/kaggle/input/competitions/cohort-x-task-2/Task_2.xlsx"),
        Path("/kaggle/input/cohort-x-task-2/Task_2.xlsx"),
        Path("/Users/nnhzzz/Downloads/cohort-x-task-2/Task_2.xlsx"),
        Path(__file__).resolve().parent / "Task_2.xlsx" if "__file__" in globals() else Path("Task_2.xlsx"),
        Path("Task_2.xlsx"),
    ]
    for p in candidates:
        if p.exists(): return p
    raise FileNotFoundError(f"Task_2.xlsx not found in any of: {candidates}")

XLSX = _discover_xlsx()
ON_KAGGLE = Path("/kaggle/working").exists()
ROOT = Path("/kaggle/working") if ON_KAGGLE else XLSX.parent
SUB_CSV = Path(os.environ.get("COHORTX_OUT", ROOT / "submission.csv"))
CACHE_DIR = ROOT / ".cache"; CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _discover_model() -> str:
    if env := os.environ.get("COHORTX_MODEL"):
        path = Path(env)
        if not path.exists():
            raise FileNotFoundError(f"COHORTX_MODEL does not exist: {path}")
        return str(path)

    local = Path(
        "/Users/nnhzzz/.lmstudio/models/lmstudio-community/"
        "Qwen3.5-9B-GGUF/Qwen3.5-9B-Q8_0.gguf"
    )
    if local.exists():
        return str(local)

    ggufs = list(Path("/kaggle/input").rglob("*.gguf")) if ON_KAGGLE else []
    preferred = [
        p for p in ggufs
        if "qwen3.5" in p.name.lower() and "9b" in p.name.lower()
    ]
    if len(preferred) == 1:
        return str(preferred[0])
    if not preferred:
        raise FileNotFoundError(
            "No Qwen3.5-9B GGUF found under /kaggle/input. Attach a Kaggle "
            "Dataset containing Qwen3.5-9B-Q8_0.gguf, or set COHORTX_MODEL."
        )
    raise RuntimeError(
        "Multiple Qwen3.5-9B GGUF files found; set COHORTX_MODEL explicitly: "
        + ", ".join(map(str, preferred))
    )

MODEL_PATH = _discover_model()
if ON_KAGGLE:
    os.environ.setdefault("COHORTX_N_GPU_LAYERS", "0")

print(f"[paths] XLSX={XLSX}  SUB={SUB_CSV}  MODEL={MODEL_PATH}  (kaggle={ON_KAGGLE})")

EM = "—"           # em dash
SEP = f" {EM} "         # required separator inside a triple
TRIPLE_DELIM = "\n\n"   # required separator between triples
TRIPLE_RE = re.compile(r'"([^"]+)"\s*—\s*"([^"]+)"\s*—\s*"([^"]+)"')
# LLM output must be stricter than label parsing.  In particular, never let a
# regex span prose/newlines looking for the next three quoted fragments (this
# corrupted one row of the first 0.72 submission with Qwen's reasoning text).
STRICT_TRIPLE_RE = re.compile(
    r'(?m)^\s*(?:(?:[-*•]|\d+[.)])\s+)?'
    r'"([^"\r\n]+)"\s*—\s*"([^"\r\n]+)"\s*—\s*"([^"\r\n]+)"\s*$'
)

[paths] XLSX=/kaggle/input/competitions/cohort-x-task-2/Task_2.xlsx  SUB=/kaggle/working/submission.csv  MODEL=/kaggle/input/datasets/nnhzzz/qwen3-5-9b-q8-0-gguf/Qwen3.5-9B-Q8_0.gguf  (kaggle=True)


## 1. Load data

In [2]:
def load_xlsx(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    def df(name):
        rows = [r[:2] for r in wb[name].iter_rows(values_only=True)]
        header, body = rows[0], [r for r in rows[1:] if any(c not in (None, "") for c in r)]
        return pd.DataFrame(body, columns=list(header))
    return df("Train"), df("Test")

train_df, test_df = load_xlsx(XLSX)
print(f"train: {len(train_df)} rows | test: {len(test_df)} rows")
print("columns:", list(train_df.columns))

def _discover_base_submission() -> Path:
    if env := os.environ.get("COHORTX_BASE_SUBMISSION"):
        path = Path(env)
        if not path.exists():
            raise FileNotFoundError(f"COHORTX_BASE_SUBMISSION does not exist: {path}")
        return path
    if ON_KAGGLE:
        input_root = Path("/kaggle/input")
        preferred = list(input_root.rglob("submission_081.csv"))
        if len(preferred) == 1:
            return preferred[0]
        if len(preferred) > 1:
            raise FileNotFoundError(
                "Multiple submission_081.csv files are attached; set "
                "COHORTX_BASE_SUBMISSION to the exact scored file: "
                f"{preferred}"
            )

        # Kaggle Dataset uploads commonly retain the original `submission.csv`
        # filename. Auto-select it only when its content uniquely identifies the
        # scored run: exact Test order/text and the five known skeleton counts.
        all_csvs = list(input_root.rglob("*.csv"))
        valid = []
        for path in all_csvs:
            try:
                candidate = pd.read_csv(path, dtype=str, keep_default_na=False)
                if len(candidate) != len(test_df):
                    continue
                if not {"eligibility_criteria", "structured"}.issubset(candidate.columns):
                    continue
                if any(
                    candidate.iloc[i]["eligibility_criteria"]
                    != test_df.iloc[i]["eligibility_criteria"]
                    for i in range(len(test_df))
                ):
                    continue
                tail_counts = [
                    len(TRIPLE_RE.findall(candidate.iloc[i]["structured"]))
                    for i in range(45, 50)
                ]
                if tail_counts == [11, 9, 31, 14, 6]:
                    valid.append(path)
            except Exception:
                continue
        if len(valid) == 1:
            print(f"[base auto-discovery] using scored CSV: {valid[0]}")
            return valid[0]
        raise FileNotFoundError(
            "The exact CSV that scored 0.81 is not attached. Download that "
            "notebook's /kaggle/working/submission.csv, upload it as a private "
            "Kaggle Dataset, attach the Dataset here, then Run All again. It may "
            "be named submission.csv or submission_081.csv. Alternatively set "
            "COHORTX_BASE_SUBMISSION to its mounted path. "
            f"Attached CSV files inspected: {all_csvs}; valid scored files: {valid}"
        )
    local_081 = XLSX.parent / "submission_081.csv"
    return local_081 if local_081.exists() else XLSX.parent / "submission.csv"

BASE_SUBMISSION = _discover_base_submission()
_base_df = pd.read_csv(BASE_SUBMISSION, dtype=str, keep_default_na=False)
if len(_base_df) != len(test_df):
    raise ValueError(
        f"base submission has {len(_base_df)} rows, expected {len(test_df)}"
    )
if not {"eligibility_criteria", "structured"}.issubset(_base_df.columns):
    raise ValueError("base submission must contain eligibility_criteria and structured")
for i in range(len(test_df)):
    if _base_df.iloc[i]["eligibility_criteria"] != test_df.iloc[i]["eligibility_criteria"]:
        raise ValueError(f"base submission row {i} does not match Test order/text")
BASE_KEEP_ROWS = frozenset(range(45))
BASE_PREDICTIONS = {
    i: _base_df.iloc[i]["structured"] for i in BASE_KEEP_ROWS
}
BASE_SHA256 = hashlib.sha256(BASE_SUBMISSION.read_bytes()).hexdigest()
print(
    f"[base 0.81] {BASE_SUBMISSION} | sha256={BASE_SHA256[:12]} | "
    "preserving rows 0-44; regenerating rows 45-49"
)

train: 100 rows | test: 50 rows
columns: ['eligibility_criteria', 'structured']
[base 0.81] /kaggle/input/datasets/nnhzzz/submission-081/submission_081.csv | sha256=18a6c05a9c63 | preserving rows 0-44; regenerating rows 45-49


## 2. Parse triples from train and inventory predicate vocabulary

In [3]:
def parse_triples(text: str) -> list[tuple[str, str, str]]:
    if not isinstance(text, str): return []
    return [(s.strip(), p.strip(), o.strip()) for s, p, o in TRIPLE_RE.findall(text)]

for _i in BASE_KEEP_ROWS:
    _segments = BASE_PREDICTIONS[_i].split(TRIPLE_DELIM)
    if not _segments or any(not STRICT_TRIPLE_RE.fullmatch(x) for x in _segments):
        raise ValueError(f"base 0.81 row {_i} is not in strict triple format")

_base_tail_counts = [
    len(parse_triples(_base_df.iloc[i]["structured"])) for i in range(45, 50)
]
_expected_fallback_counts = [11, 9, 31, 14, 6]
if ON_KAGGLE and _base_tail_counts != _expected_fallback_counts:
    raise ValueError(
        "submission_081.csv does not look like the scored fallback run: "
        f"tail counts={_base_tail_counts}, expected={_expected_fallback_counts}"
    )
print(f"[base 0.81] verified tail fallback counts: {_base_tail_counts}")

train_df["triples"] = train_df["structured"].map(parse_triples)
all_triples = [t for ts in train_df["triples"] for t in ts]
print(f"total train triples: {len(all_triples)}")
print(f"per-study: min={train_df['triples'].map(len).min()}, "
      f"median={int(train_df['triples'].map(len).median())}, "
      f"max={train_df['triples'].map(len).max()}, "
      f"mean={train_df['triples'].map(len).mean():.1f}")

pred_counts = Counter(p for _, p, _ in all_triples)
subj_counts = Counter(s for s, _, _ in all_triples)
print(f"unique predicates: {len(pred_counts)} (top 8: {pred_counts.most_common(8)})")
print(f"unique subjects:   {len(subj_counts)} (top 6: {subj_counts.most_common(6)})")

[base 0.81] verified tail fallback counts: [11, 9, 31, 14, 6]
total train triples: 2894
per-study: min=6, median=24, max=104, mean=28.9
unique predicates: 877 (top 8: [('includes criterion', 1116), ('excludes patients with', 108), ('has inclusion criteria', 103), ('has exclusion criteria', 103), ('age', 38), ('consent', 27), ('excludes patient with', 24), ('imaging modality', 14)])
unique subjects:   161 (top 6: [('Exclusion Criteria Set', 491), ('Inclusion Criteria Set', 251), ('Study', 202), ('Exclusion Criteria', 152), ('Inclusion Criterion 1', 149), ('Inclusion Criterion 2', 136)])


## 3. Rule-based skeleton generator

Most outputs have a predictable backbone given the section structure:
- `Study — has inclusion criteria — Inclusion Criteria`
- `Study — has exclusion criteria — Exclusion Criteria`
- `Inclusion Criteria — includes criterion — Inclusion Criterion N`  (one per criterion)
- `Exclusion Criteria — includes criterion — Exclusion Criterion N`

We generate this as a fallback. It is not guaranteed: nested lists,
free-form clauses, named subgroups, and multi-cohort studies make criterion
boundaries annotation-dependent, so a coherent model-generated structure is
preferred during post-processing.

In [4]:
INC_RE = re.compile(
    r'(?im)^[\s>*\-]*(?:\d+(?:\.\d+)*\.?\s+)?inclusion\s+criteria\b\s*:?\s*'
)
EXC_RE = re.compile(
    r'(?im)^[\s>*\-]*(?:\d+(?:\.\d+)*\.?\s+)?exclusion\s+criteria\b\s*:?\s*'
)

def split_sections(text: str) -> tuple[str, str]:
    """Return (inclusion_text, exclusion_text) extracted from raw input."""
    inc_m = INC_RE.search(text); exc_m = EXC_RE.search(text)
    inc_text = exc_text = ""
    if inc_m and exc_m:
        if inc_m.start() < exc_m.start():
            inc_text = text[inc_m.end():exc_m.start()]
            exc_text = text[exc_m.end():]
        else:
            exc_text = text[exc_m.end():inc_m.start()]
            inc_text = text[inc_m.end():]
    elif inc_m:
        inc_text = text[inc_m.end():]
    elif exc_m:
        exc_text = text[exc_m.end():]
    else:
        inc_text = text   # no header — treat all as inclusion
    return inc_text.strip(), exc_text.strip()

ITEM_MARKER_RE = re.compile(
    r'(?m)^(?P<indent>[ \t]*)(?P<marker>'
    r'\\?[-*•◦●▪]|'
    r'(?:\d+(?:\.\d+)*|[ivxIVX]+|[A-Za-z])\\?[.)]|'
    r'\(\d+\)'
    r')[ \t]+'
)
CONTAINER_PRELUDE_RE = re.compile(
    r'(?i)\b(?:criteria|following|conditions|requirements)\s*:?[ \t]*$'
)

def split_items(section_text: str) -> list[str]:
    """Split top-level criteria while retaining nested text inside its parent.

    Treating every nested bullet as a new criterion caused systematic
    over-numbering. Selecting the minimum indentation level raises exact
    inclusion/exclusion structural-count agreement on Train from 68% to 75%.
    """
    if not section_text.strip():
        return []
    markers = list(ITEM_MARKER_RE.finditer(section_text))
    if not markers:
        parts = re.split(r'\n\s*\n|\n', section_text)
        return [re.sub(r'\s+', ' ', x).strip() for x in parts if len(x.strip()) > 2]

    by_indent: dict[int, list[re.Match]] = {}
    for marker in markers:
        indent = len(marker.group("indent").expandtabs(4))
        by_indent.setdefault(indent, []).append(marker)
    depths = sorted(by_indent)
    chosen = by_indent[depths[0]]
    leading_item: str | None = None
    if len(chosen) == 1 and len(depths) > 1 and len(by_indent[depths[1]]) >= 2:
        children = by_indent[depths[1]]
        prelude = section_text[chosen[0].end():children[0].start()].strip()
        chosen = children
        if prelude and not CONTAINER_PRELUDE_RE.search(prelude):
            leading_item = prelude

    items: list[str] = []
    if leading_item:
        items.append(leading_item)
    for j, marker in enumerate(chosen):
        end = chosen[j + 1].start() if j + 1 < len(chosen) else len(section_text)
        items.append(section_text[marker.end():end].strip())
    return [
        re.sub(r'\s+', ' ', item).strip()
        for item in items if len(item.strip()) > 2
    ]

def make_skeleton(text: str, *, inc_label: str = "Inclusion Criteria",
                  exc_label: str = "Exclusion Criteria") -> tuple[list[tuple[str, str, str]], list[str], list[str]]:
    """Return (backbone_triples, inclusion_items, exclusion_items)."""
    inc_txt, exc_txt = split_sections(text)
    inc_items = split_items(inc_txt)
    exc_items = split_items(exc_txt)
    triples: list[tuple[str, str, str]] = []
    if inc_items:
        triples.append(("Study", "has inclusion criteria", inc_label))
    if exc_items:
        triples.append(("Study", "has exclusion criteria", exc_label))
    for i in range(len(inc_items)):
        triples.append((inc_label, "includes criterion", f"Inclusion Criterion {i+1}"))
    for i in range(len(exc_items)):
        triples.append((exc_label, "includes criterion", f"Exclusion Criterion {i+1}"))
    return triples, inc_items, exc_items

# sanity-check on the first 3 train rows
for i in range(3):
    sk, inc, exc = make_skeleton(train_df.iloc[i]["eligibility_criteria"])
    print(f"row {i}: skeleton={len(sk)} | inc_items={len(inc)} | exc_items={len(exc)}")

row 0: skeleton=15 | inc_items=5 | exc_items=8
row 1: skeleton=26 | inc_items=4 | exc_items=20
row 2: skeleton=4 | inc_items=1 | exc_items=1


## 4. TF-IDF retrieval — pick the most similar train rows for each test row

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

WORD_VEC = TfidfVectorizer(
    ngram_range=(1, 2), min_df=1, max_df=0.95, sublinear_tf=True,
)
CHAR_VEC = TfidfVectorizer(
    # LOO on the 100 supplied labels favours shorter character fragments and
    # keeping rare medical strings. No external corpus is used.
    analyzer="char_wb", ngram_range=(2, 5), min_df=1, sublinear_tf=True,
)
_train_texts = train_df["eligibility_criteria"].tolist()
_train_word_mat = WORD_VEC.fit_transform(_train_texts)
_train_char_mat = CHAR_VEC.fit_transform(_train_texts)
_train_word_counts = np.asarray([max(1, len(re.findall(r"\w+", x))) for x in _train_texts])

def _shape_features(text: str) -> np.ndarray:
    inc_text, exc_text = split_sections(text)
    inc_items, exc_items = split_items(inc_text), split_items(exc_text)
    words = re.findall(r"\w+", text)
    return np.asarray([
        len(inc_items), len(exc_items), len(words), len(text), text.count("\n"),
        len(re.findall(r"(?m)^\s*\*", text)),
        len(re.findall(r"(?m)^\s*\d+[.)]", text)),
        len(re.findall(r"(?m)^\s*(?:[-*•]|\\-)", text)),
        np.mean([len(re.findall(r"\w+", x)) for x in inc_items]) if inc_items else 0,
        np.mean([len(re.findall(r"\w+", x)) for x in exc_items]) if exc_items else 0,
    ], dtype=float)

_shape_scaler = StandardScaler()
_train_shape_mat = _shape_scaler.fit_transform(
    np.log1p(np.vstack([_shape_features(x) for x in _train_texts]))
)

def _shape_similarities(query: str) -> np.ndarray:
    q = _shape_scaler.transform(np.log1p(_shape_features(query))[None, :])[0]
    mean_sq_distance = np.mean((_train_shape_mat - q) ** 2, axis=1)
    return np.exp(-mean_sq_distance / 2)

def _annotation_regime(triples: list[tuple[str, str, str]]) -> str:
    """Infer the two dominant annotation styles from Train labels.

    The competition labels contain a concise ``Criteria Set`` style and a
    denser plain ``Criteria`` style. Mixing their exemplars makes the model
    imitate conflicting subject names, predicate granularity and row counts.
    """
    standard_parents = {
        "Inclusion Criteria", "Exclusion Criteria",
        "Inclusion Criteria Set", "Exclusion Criteria Set",
    }
    backbone_objects = [
        o for s, p, o in triples
        if p in {"has inclusion criteria", "has exclusion criteria"}
    ]
    if any(x not in standard_parents for x in backbone_objects):
        return "other"
    structural_fields = [
        field
        for s, p, o in triples
        if p in {"has inclusion criteria", "has exclusion criteria", "includes criterion"}
        for field in (s, o)
    ]
    if any(re.search(r"(?:Inclusion|Exclusion) Criteria Set$", x) for x in structural_fields):
        return "set"
    if any(re.search(r"(?:Inclusion|Exclusion) Criteria$", x) for x in structural_fields):
        return "plain"
    return "other"

_train_regimes = np.asarray([_annotation_regime(ts) for ts in train_df["triples"]])
print("annotation regimes:", Counter(_train_regimes))

# A class-balanced character model recovers the minority dense/plain regime
# much better than kNN in strict LOO (balanced accuracy .798 vs .713, overall
# accuracy .854 vs .844). KNN remains the honest fallback for held-out local
# evaluations where the queried train label must be excluded.
_conventional_regime_idxs = np.where(_train_regimes != "other")[0]
REGIME_VEC = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), min_df=1, sublinear_tf=True,
)
_regime_x = REGIME_VEC.fit_transform([
    _train_texts[i] for i in _conventional_regime_idxs
])
_regime_y = np.asarray([
    1 if _train_regimes[i] == "plain" else 0
    for i in _conventional_regime_idxs
])
REGIME_MODEL = LogisticRegression(
    C=0.1, class_weight="balanced", solver="liblinear",
    max_iter=2000, random_state=20260714,
).fit(_regime_x, _regime_y)

def _retrieval_scores(query: str) -> tuple[np.ndarray, np.ndarray]:
    """Return final and base hybrid scores for one source document."""
    word_sims = cosine_similarity(WORD_VEC.transform([query]), _train_word_mat)[0]
    char_sims = cosine_similarity(CHAR_VEC.transform([query]), _train_char_mat)[0]
    lexical = 0.25 * word_sims + 0.75 * char_sims
    base = 0.90 * lexical + 0.10 * _shape_similarities(query)
    query_words = max(1, len(re.findall(r"\w+", query)))
    # A small log-length penalty improved LOO label similarity and prevents a
    # short test row from retrieving 80-90-triple demonstrations.
    length_penalty = 0.05 * np.abs(np.log(query_words / _train_word_counts))
    return base - length_penalty, base

def predict_annotation_regime(query: str, *, exclude_idx: int | None = None) -> str:
    """Predict Set/plain style; use label-excluding kNN for held-out rows."""
    if exclude_idx is None:
        pred = int(REGIME_MODEL.predict(REGIME_VEC.transform([query]))[0])
        return "plain" if pred else "set"

    # Honest held-out path used by local ablations and count-model features.
    _, base = _retrieval_scores(query)
    if exclude_idx is not None:
        base[exclude_idx] = -np.inf
    ranked = [i for i in base.argsort()[::-1] if _train_regimes[i] != "other"][:3]
    votes = Counter(_train_regimes[ranked])
    if not votes:
        return "set"
    best_count = max(votes.values())
    tied = {name for name, n in votes.items() if n == best_count}
    return str(next(_train_regimes[i] for i in ranked if _train_regimes[i] in tied))

def retrieve_examples(query: str, k: int = 3, *, exclude_idx: int | None = None) -> list[int]:
    """Length- and annotation-regime-aware hybrid lexical retrieval.

    Character n-grams are robust to medical morphology, abbreviations and noisy
    punctuation; word n-grams retain phrase-level specificity.  A 75/25 blend
    beat the prior word-only retriever on held-out label/predicate similarity.
    """
    sims, _ = _retrieval_scores(query)
    if exclude_idx is not None:
        sims[exclude_idx] = -np.inf
    regime = predict_annotation_regime(query, exclude_idx=exclude_idx)
    global_ranked = [i for i in sims.argsort()[::-1] if i != exclude_idx]
    # Four train rows use study-specific parent names. Keep one when it is a
    # very close domain match (notably CRC / knee-stiffness test cases), rather
    # than discarding the most medically relevant demonstration merely because
    # it is outside the two dominant serialization regimes.
    special = [
        i for i in global_ranked[:5]
        if _train_regimes[i] == "other" and sims[i] >= 0.85 * sims[global_ranked[0]]
    ][:1]
    ranked = special + [
        i for i in sims.argsort()[::-1]
        if _train_regimes[i] == regime and i != exclude_idx and i not in special
    ]
    # Defensive fallback only; each dominant regime has far more than k rows.
    if len(ranked) < k:
        ranked.extend(
            i for i in sims.argsort()[::-1]
            if i not in ranked and i != exclude_idx
        )
    return [int(i) for i in ranked[:k]]

# Eyeball one retrieval
demo_idx = retrieve_examples(test_df.iloc[0]["eligibility_criteria"], k=3)
print("test[0] nearest train idx:", demo_idx)

annotation regimes: Counter({np.str_('set'): 71, np.str_('plain'): 25, np.str_('other'): 4})
test[0] nearest train idx: [60, 26, 53]


## 5. Train-only guidance: criterion memory + output-count calibration

Row retrieval teaches the overall schema, while this small memory retrieves
predicate *names* for each parsed criterion. It never supplies an object/fact,
so it cannot leak another study's age, diagnosis or threshold.

In [6]:
STRUCTURAL_PREDICATES_EARLY = {
    "has inclusion criteria", "has exclusion criteria", "includes criterion",
}
CRITERION_SUBJECT_RE = re.compile(r"^(Inclusion|Exclusion) Criterion (\d+)$")

def _build_criterion_bank() -> list[dict]:
    bank: list[dict] = []
    for row_idx, row in train_df.iterrows():
        inc_text, exc_text = split_sections(row["eligibility_criteria"])
        sections = {
            "Inclusion": split_items(inc_text),
            "Exclusion": split_items(exc_text),
        }
        triples = row["triples"]
        for kind, items in sections.items():
            numbered = [
                int(m.group(2)) for s, _, _ in triples
                if (m := CRITERION_SUBJECT_RE.fullmatch(s)) and m.group(1) == kind
            ]
            max_number = max(numbered, default=0)
            # Only exact alignments are safe enough to map source item N to the
            # predicates of gold subject Criterion N.
            if not items or len(items) != max_number:
                continue
            for n, item in enumerate(items, 1):
                subject = f"{kind} Criterion {n}"
                predicates = list(dict.fromkeys(
                    p for s, p, _ in triples
                    if s == subject and p not in STRUCTURAL_PREDICATES_EARLY
                ))
                if predicates:
                    bank.append({
                        "kind": kind, "text": item, "predicates": predicates,
                        "row_idx": int(row_idx), "criterion_no": n,
                    })
    return bank

CRITERION_BANK = _build_criterion_bank()
_micro_texts = [x["text"] for x in CRITERION_BANK]
MICRO_WORD_VEC = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
MICRO_CHAR_VEC = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True,
)
_micro_word_mat = MICRO_WORD_VEC.fit_transform(_micro_texts)
_micro_char_mat = MICRO_CHAR_VEC.fit_transform(_micro_texts)
_micro_kind_indices = {
    kind: np.asarray([i for i, x in enumerate(CRITERION_BANK) if x["kind"] == kind])
    for kind in ("Inclusion", "Exclusion")
}
print(f"criterion predicate memory: {len(CRITERION_BANK)} aligned items")

def criterion_predicate_hints(
    criteria_text: str, *, top_predicates: int = 3,
    exclude_row_idx: int | None = None,
) -> list[str]:
    """Return compact, source-item-aligned predicate suggestions."""
    inc_text, exc_text = split_sections(criteria_text)
    result: list[str] = []
    for kind, items in (
        ("Inclusion", split_items(inc_text)),
        ("Exclusion", split_items(exc_text)),
    ):
        bank_idxs = _micro_kind_indices[kind]
        if not len(bank_idxs):
            continue
        # Cap only the optional hint block, never source processing/generation.
        for criterion_no, item in enumerate(items[:40], 1):
            word = cosine_similarity(
                MICRO_WORD_VEC.transform([item]), _micro_word_mat[bank_idxs]
            )[0]
            char = cosine_similarity(
                MICRO_CHAR_VEC.transform([item]), _micro_char_mat[bank_idxs]
            )[0]
            sims = 0.25 * word + 0.75 * char
            local_top = sims.argsort()[::-1][:20]
            votes: Counter = Counter()
            for local_i in local_top:
                entry = CRITERION_BANK[int(bank_idxs[local_i])]
                if exclude_row_idx is not None and entry["row_idx"] == exclude_row_idx:
                    continue
                weight = float(sims[local_i]) / max(1, len(entry["predicates"]))
                for predicate in entry["predicates"]:
                    votes[predicate] += weight
            predicates = [p for p, _ in votes.most_common(top_predicates)]
            if predicates:
                result.append(
                    f"{kind} Criterion {criterion_no}: " + " | ".join(predicates)
                )
    return result

# A train-only random forest estimates *rough* output complexity. It is prompt
# guidance and a diagnostic, never a hard trimming rule.
from sklearn.ensemble import RandomForestRegressor

def _count_features(text: str, regime: str) -> np.ndarray:
    shape = _shape_features(text)
    extra = np.asarray([
        text.count(";"), text.count(","), text.count(":"), text.count("("),
        len(re.findall(r"(?i)\b(?:and|or|unless|except|including)\b", text)),
        len(re.findall(r"\d+(?:\.\d+)?", text)),
        float(regime == "set"), float(regime == "plain"), float(regime == "other"),
    ])
    return np.concatenate([np.log1p(shape), np.log1p(extra[:6]), extra[6:]])

_loo_train_regimes = [
    predict_annotation_regime(text, exclude_idx=i)
    for i, text in enumerate(_train_texts)
]
_count_x = np.vstack([
    _count_features(text, regime)
    for text, regime in zip(_train_texts, _loo_train_regimes)
])
_count_y = train_df["triples"].map(len).to_numpy()
COUNT_MODEL = RandomForestRegressor(
    n_estimators=300, min_samples_leaf=3, max_features=0.8,
    random_state=20260714, n_jobs=-1,
).fit(_count_x, _count_y)

def expected_triple_count(criteria_text: str, regime: str | None = None) -> tuple[int, int, int]:
    regime = regime or predict_annotation_regime(criteria_text)
    center = int(round(float(COUNT_MODEL.predict(
        _count_features(criteria_text, regime)[None, :]
    )[0])))
    return center, max(6, center - 10), center + 10

criterion predicate memory: 861 aligned items


## 6. Prompt template

Few-shot ICL is the right approach here: 100 train rows is tiny, and the
output's format/structure dominates the answer. The prompt:
  * spells out the exact format (em-dash, quotes, blank-line separator),
  * gives explicit list-count/style guidance learned from Train,
  * provides 2 retrieved training (input, output) pairs plus per-item predicate hints,
  * asks the model to emit only the complete structured output.

In [7]:
TOP_LEAF_PREDICATES = [
    p for p, _ in pred_counts.most_common()
    if p not in STRUCTURAL_PREDICATES_EARLY
][:35]

SYSTEM_PROMPT = """You convert free-text clinical-trial eligibility criteria into a list of structured triples.

Each triple is on its own line in EXACTLY this form:
"Subject" — "predicate" — "object"

Hard formatting rules:
1. The separator between subject, predicate, and object is " — " — a space, the EM-DASH character (U+2014), then a space. NEVER use a hyphen "-" or en-dash "–".
2. All three fields are wrapped in straight double quotes ".
3. Triples are separated by ONE blank line ("\\n\\n").

Content rules:
- Begin with the backbone triples: ("Study" — "has inclusion criteria" — the requested inclusion parent), ("Study" — "has exclusion criteria" — the requested exclusion parent), then (parent — "includes criterion" — "Inclusion Criterion N" / "Exclusion Criterion N") for each source item.
- Then emit detail triples whose subject is "Inclusion Criterion N" / "Exclusion Criterion N". The predicate is a short medical-domain phrase ("age", "procedure", "diagnosis", "BMI", "excludes patients with", "consent", "imaging modality", "infection", "pregnancy", "history", "consent status", etc.). The object is the short medically-meaningful value from the source.
- If the source defines or elaborates a downstream concept (e.g. "Definition of ACS:" or "confirmed by ..."), add extra triples whose subject IS that concept (e.g. "Acute myocardial infarction" — "meets definition if" — "...").
- Paraphrase as little as possible — keep the medical wording.
- Never copy a fact, number, age, diagnosis, or procedure from an example unless it is explicitly supported by the final user's source text.
- Predicate hints are optional vocabulary suggestions, not facts. Use another concise predicate whenever the source calls for it; the vocabulary is open.
- Cover every source criterion. Do not invent a criterion, number, threshold, diagnosis, treatment, or fact.

Common train predicate vocabulary (non-exhaustive):
""" + ", ".join(TOP_LEAF_PREDICATES) + """

OUTPUT FORMAT: Output ONLY the list of triples. No preamble, no explanation, no headings, no chain-of-thought, no markdown fences. Start your reply with the very first triple character, which is the opening double quote of "Study"."""

def format_skeleton(skeleton: list[tuple[str, str, str]]) -> str:
    return TRIPLE_DELIM.join(f'"{s}"{SEP}"{p}"{SEP}"{o}"' for s, p, o in skeleton)

def _final_user_prompt(
    criteria_text: str, regime: str, *, exclude_train_idx: int | None = None,
    has_custom_exemplar: bool = False,
) -> str:
    center, lower, upper = expected_triple_count(criteria_text, regime)
    hints = criterion_predicate_hints(criteria_text, exclude_row_idx=exclude_train_idx)
    inc_text, exc_text = split_sections(criteria_text)
    n_inc, n_exc = len(split_items(inc_text)), len(split_items(exc_text))
    all_inc = max(n_inc, len(list(ITEM_MARKER_RE.finditer(inc_text))))
    all_exc = max(n_exc, len(list(ITEM_MARKER_RE.finditer(exc_text))))
    if regime == "set":
        style = (
            'Use parent subjects "Inclusion Criteria Set" / "Exclusion Criteria Set". '
            "Prefer concise criterion-specific predicates and roughly one atomic detail "
            "triple per supported concept."
        )
    else:
        style = (
            'Use parent subjects "Inclusion Criteria" / "Exclusion Criteria". '
            "Use the denser plain-Criteria annotation style: split supported compound "
            'concepts; generic "excludes patients with" is allowed when appropriate.'
        )
    if has_custom_exemplar:
        style += (
            " A highly similar demonstration uses study-specific parent names. "
            "Use such named cohort/criteria parents only if the SOURCE clearly defines them."
        )
    hint_block = "\n".join(f"- {x}" for x in hints) if hints else "- none"
    return f"""<SOURCE>
{criteria_text.strip()}
</SOURCE>

<TRAIN_ONLY_SOFT_GUIDANCE>
Annotation style: {style}
Complexity estimate: about {center} triples (soft range {lower}-{upper}); fidelity is more important than hitting this number.
List-structure estimate: {n_inc} top-level inclusion / {n_exc} top-level exclusion items; {all_inc} / {all_exc} visible markers including nested items. Normally keep multiple facts from one bullet on the same Criterion N subject. Promote a nested marker only when it is an independently stated criterion or the SOURCE defines separate cohorts/categories.
Optional predicate-name hints aligned to parsed source items:
{hint_block}
</TRAIN_ONLY_SOFT_GUIDANCE>

Structure only the facts inside <SOURCE>. Output triples only."""

def build_messages(
    criteria_text: str, exemplar_indices: list[int], regime: str | None = None,
    *, exclude_train_idx: int | None = None,
) -> list[dict]:
    """Chat-template ICL: each exemplar is a clean user/assistant turn pair, and
    the test case is the final user turn. This keeps the model from echoing
    headings or generating a 'Thinking Process' preamble."""
    regime = regime or predict_annotation_regime(criteria_text)
    msgs: list[dict] = [{"role": "system", "content": SYSTEM_PROMPT}]
    for idx in exemplar_indices:
        row = train_df.iloc[idx]
        msgs.append({"role": "user",      "content": row["eligibility_criteria"].strip()})
        msgs.append({"role": "assistant", "content": row["structured"].strip()})
    msgs.append({
        "role": "user",
        "content": _final_user_prompt(
            criteria_text, regime, exclude_train_idx=exclude_train_idx,
            has_custom_exemplar=any(_train_regimes[i] == "other" for i in exemplar_indices),
        ),
    })
    return msgs

## 7. Load the LLM (Qwen3.5-9B-Q8, CPU-only on Kaggle)

`llama-cpp-python` 0.3.30+ is required for the qwen35 GGUF architecture. The
cell first searches attached Kaggle Datasets for an offline wheel. If none is
attached, it can install from PyPI during dependency setup; set
`COHORTX_ALLOW_PYPI=0` to enforce fully offline setup as well as inference.

In [8]:
def _ensure_llama_cpp() -> None:
    try:
        module = importlib.import_module("llama_cpp")
        version = tuple(int(x) for x in module.__version__.split(".")[:3])
        if version >= (0, 3, 30):
            print(f"llama-cpp-python {module.__version__} is ready")
            return
    except (ImportError, ValueError):
        pass

    wheels = list(Path("/kaggle/input").rglob("llama_cpp_python-*.whl")) if ON_KAGGLE else []
    if wheels:
        print(f"Installing attached offline wheel: {wheels[0].name}")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-index",
            "--no-deps", "--force-reinstall", str(wheels[0]),
        ])
    elif os.environ.get("COHORTX_ALLOW_PYPI", "1") == "1":
        print("Installing llama-cpp-python>=0.3.30 (dependency setup only)...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--upgrade",
            "llama-cpp-python>=0.3.30",
        ])
    else:
        raise RuntimeError(
            "llama-cpp-python>=0.3.30 is missing. Attach a compatible wheel "
            "Dataset or temporarily enable Internet for the setup run."
        )
    for name in list(sys.modules):
        if name == "llama_cpp" or name.startswith("llama_cpp."):
            del sys.modules[name]
    importlib.invalidate_caches()
    module = importlib.import_module("llama_cpp")
    version = tuple(int(x) for x in module.__version__.split(".")[:3])
    if version < (0, 3, 30):
        raise RuntimeError(f"llama-cpp-python {module.__version__} is too old")

_ensure_llama_cpp()
from llama_cpp import Llama

def load_llm():
    n_gpu_layers = int(os.environ.get("COHORTX_N_GPU_LAYERS", "0"))
    return Llama(
        model_path=MODEL_PATH,
        n_ctx=16384,            # headroom for 2 exemplars + guidance + dynamic output
        n_threads=os.cpu_count() or 8,
        n_gpu_layers=n_gpu_layers,
        seed=20260617,
        verbose=False,
    )

# Comment out the next line during prompt iteration to avoid the ~30 s load.
# llm = load_llm()

Installing llama-cpp-python>=0.3.30 (dependency setup only)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.5 MB/s eta 0:00:00


## 8. Single-row generation — force Qwen's documented no-think branch

In [9]:
GEN_KWARGS = dict(
    temperature=0.15,
    top_p=0.9,
    repeat_penalty=1.05,
)
MAX_OUTPUT_TOKENS = int(os.environ.get("COHORTX_MAX_TOKENS", "4096"))
MIN_OUTPUT_TOKENS = int(os.environ.get("COHORTX_MIN_TOKENS", "1024"))
# The scored run already fixes rows 0-44. Review remains off while generating
# the five missing tail rows, avoiding the 13,099-second review overhead.
REVIEW_MODE = os.environ.get("COHORTX_REVIEW", "off").lower()  # off recommended
K_SHOTS = int(os.environ.get("COHORTX_K_SHOTS", "2"))
MAX_WALL_SECONDS = int(os.environ.get("COHORTX_MAX_WALL_SECONDS", "39600"))

# Train-only, source-matched demonstrations for the five rows that timed out in
# the scored 0.81 run.  Four pairs equal the hybrid retriever's choices.  Row 46
# is the deliberate correction: train 95 covers contrast allergy + metal-image
# artifacts and train 54 covers pulmonary CT/procedure + iodine/pregnancy/consent,
# instead of the automatic but unrelated post-operative knee-stiffness example.
TAIL_EXEMPLARS = {
    45: (15, 41),
    46: (95, 54),
    47: (32, 37),
    48: (13, 54),
    49: (85, 49),
}

def _format_qwen_no_think(messages: list[dict]) -> str:
    """Render Qwen ChatML and pre-fill an empty, closed thinking block.

    llama-cpp-python 0.3.30 does not expose ``chat_template_kwargs`` through
    ``create_chat_completion``. Qwen3.5's default template otherwise opens a
    thinking block and can spend the entire output budget before producing a
    triple. This prefix is exactly the template's ``enable_thinking=false``
    branch.
    """
    prompt = "".join(
        f"<|im_start|>{m['role']}\n{m['content'].strip()}<|im_end|>\n"
        for m in messages
    )
    return prompt + "<|im_start|>assistant\n<think>\n\n</think>\n\n"

def llm_generate(
    llm: "Llama", criteria_text: str, k: int = K_SHOTS,
    *, exclude_train_idx: int | None = None, test_idx: int | None = None,
) -> tuple[str, dict]:
    forced = list(TAIL_EXEMPLARS.get(test_idx, ()))
    automatic = retrieve_examples(
        criteria_text, k=max(k, K_SHOTS), exclude_idx=exclude_train_idx,
    )
    idxs = (forced + [i for i in automatic if i not in forced])[:k]
    regime = predict_annotation_regime(criteria_text, exclude_idx=exclude_train_idx)
    center, _, _ = expected_triple_count(criteria_text, regime)
    token_budget = min(MAX_OUTPUT_TOKENS, max(MIN_OUTPUT_TOKENS, center * 36 + 256))
    msgs = build_messages(
        criteria_text, idxs, regime=regime,
        exclude_train_idx=exclude_train_idx,
    )
    prompt = _format_qwen_no_think(msgs)
    started = time.time()
    out = llm.create_completion(
        prompt=prompt,
        stop=["<|im_end|>", "<|endoftext|>"],
        max_tokens=token_budget,
        **GEN_KWARGS,
    )
    choice = out["choices"][0]
    meta = {
        "retrieval_indices": idxs,
        "annotation_regime": regime,
        "finish_reason": choice.get("finish_reason"),
        "usage": out.get("usage", {}),
        "elapsed_seconds": round(time.time() - started, 3),
        "prompt_chars": len(prompt),
        "expected_triples": center,
        "max_tokens": token_budget,
    }
    return choice["text"], meta

REVIEW_INSTRUCTION = """Audit your previous candidate against the SOURCE and return the COMPLETE corrected triple list.

Checklist:
1. Preserve the requested Criteria vs Criteria Set parent style.
2. Cover every source criterion, but do not create extra Criterion N nodes just because one bullet contains multiple facts; attach those facts to the same subject. Respect genuine nested cohorts/categories.
3. Add medically meaningful facts that the candidate missed. Remove or correct anything not explicitly supported by SOURCE, especially copied numbers, ages, diagnoses, thresholds or procedures.
4. Prefer the train predicate vocabulary hints where semantically accurate, but the predicate vocabulary remains open.
5. Output only the complete corrected triples in the exact required format. No audit notes, reasoning, headings or markdown."""

def llm_review(
    llm: "Llama", criteria_text: str, candidate: str, *,
    regime: str, exclude_train_idx: int | None = None,
) -> tuple[str, dict]:
    """A no-think source-grounded second pass; no extra dataset or model."""
    center, _, _ = expected_triple_count(criteria_text, regime)
    token_budget = min(MAX_OUTPUT_TOKENS, max(MIN_OUTPUT_TOKENS, center * 38 + 256))
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": _final_user_prompt(
            criteria_text, regime, exclude_train_idx=exclude_train_idx,
        )},
        {"role": "assistant", "content": candidate},
        {"role": "user", "content": REVIEW_INSTRUCTION},
    ]
    prompt = _format_qwen_no_think(messages)
    started = time.time()
    generation = dict(GEN_KWARGS)
    generation["temperature"] = 0.05
    out = llm.create_completion(
        prompt=prompt,
        stop=["<|im_end|>", "<|endoftext|>"],
        max_tokens=token_budget,
        **generation,
    )
    choice = out["choices"][0]
    return choice["text"], {
        "review_finish_reason": choice.get("finish_reason"),
        "review_usage": out.get("usage", {}),
        "review_elapsed_seconds": round(time.time() - started, 3),
        "review_prompt_chars": len(prompt),
        "review_max_tokens": token_budget,
    }

## 9. Post-processing — enforce the exact submission format

Most failure modes from leaderboard discussion are formatting bugs (wrong
dash, no blank line, lost quotes, embedded chain-of-thought). We:
 - strip any prose around the triples,
 - normalize all dashes to U+2014 with single spaces,
 - guarantee triples are exactly `\n\n`-separated,
 - drop malformed lines that don't parse as a triple.

In [10]:
# Triples to reject — generic placeholders that leak from the format spec /
# the model's "let me show the schema" style. Spotted in 8/8 early outputs.
PLACEHOLDERS = {
    "Subject", "subject", "Predicate", "predicate", "Object", "object",
    "S", "P", "O", "N", "X", "Y",
}
PLACEHOLDER_TAIL = re.compile(r"\b(Criterion|Criteria)\s+N\b", re.IGNORECASE)
BACKBONE_PREDICATES = {"has inclusion criteria", "has exclusion criteria"}
STRUCTURAL_PREDICATES = BACKBONE_PREDICATES | {"includes criterion"}
PROMPT_LEAK_RE = re.compile(
    r"(?i)(?:u\+2014|em[ -]?dash|en[ -]?dash|blank line|opening double quote|"
    r"output format|subject\s*[—-]\s*predicate\s*[—-]\s*object|"
    r"never use a hyphen|triple character)"
)
REDUNDANT_META_PREDICATE_RE = re.compile(r"(?i)\bsynonyms?\b")

NUMBER_WORDS = {
    "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4,
    "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9,
    "ten": 10, "eleven": 11, "twelve": 12, "thirteen": 13,
    "fourteen": 14, "fifteen": 15, "sixteen": 16,
    "seventeen": 17, "eighteen": 18, "nineteen": 19,
    "twenty": 20, "thirty": 30, "forty": 40, "fifty": 50,
    "sixty": 60, "seventy": 70, "eighty": 80, "ninety": 90,
}
# Allow clinical compact forms (`1.5mg`, `16w0d`, even source typo
# `between18`) but do not treat the 1 in uppercase biomarker `FEV1`/`HbA1c`
# as a threshold.
NUMBER_RE = re.compile(r"(?<![A-Z0-9])\d+(?:\.\d+)?")
SOURCE_STOPWORDS = set("""
a an the and or of to in on with without for from by as at is are be been being
has have had any all no not other than within during prior before after per via
only if who that this these those their its into due such including according
based able must may should can will status patient patients subject subjects
criterion criteria study
""".split())

def _strip_list_numbers(text: str) -> str:
    return re.sub(
        r"(?m)^[ \t]*(?:\d+(?:\.\d+)*\\?[.)]|\(\d+\))[ \t]+", "", text
    )

def _numeric_tokens(text: str, *, source: bool = False) -> list[tuple[float, str]]:
    if source:
        text = _strip_list_numbers(text)
    found: list[tuple[float, str]] = []
    for raw in NUMBER_RE.findall(text):
        try:
            found.append((float(raw), raw))
        except ValueError:
            pass
    for word, value in NUMBER_WORDS.items():
        if re.search(rf"(?i)\b{word}\b", text):
            found.append((float(value), word))
    # Common clinical paraphrase seen in Train.
    if re.search(r"(?i)\bhalf (?:a |one )?year\b", text):
        found.append((6.0, "half-year"))
    return found

def _numeric_object_supported(obj: str, source_text: str) -> bool:
    obj_nums = _numeric_tokens(obj)
    if not obj_nums:
        return True
    src_nums = _numeric_tokens(source_text, source=True)
    for obj_value, obj_raw in obj_nums:
        supported = False
        for src_value, src_raw in src_nums:
            if abs(obj_value - src_value) < 1e-9:
                supported = True; break
            # Annotation occasionally renders `<70` as an inclusive integer
            # endpoint 69, or compacts gestational `16+0` as `160`.
            if obj_value.is_integer() and src_value.is_integer() and abs(obj_value - src_value) == 1:
                supported = True; break
            if obj_raw.isdigit() and src_raw.isdigit() and (
                (len(obj_raw) >= 2 and obj_raw in src_raw)
                or (len(src_raw) >= 2 and src_raw in obj_raw)
                or (len(obj_raw) == 1 and len(src_raw) == 3 and src_raw.endswith(obj_raw))
            ):
                supported = True; break
        if not supported:
            return False
    return True

def _content_tokens(text: str) -> set[str]:
    tokens = set()
    for token in re.findall(r"[a-z0-9]+", text.lower()):
        if len(token) <= 2 or token.isdigit() or token in SOURCE_STOPWORDS:
            continue
        # Lightweight morphology normalization without an external corpus.
        if len(token) > 5 and token.endswith("ies"):
            token = token[:-3] + "y"
        elif len(token) > 5 and token.endswith("ing"):
            token = token[:-3]
        elif len(token) > 4 and token.endswith("ed"):
            token = token[:-2]
        elif len(token) > 4 and token.endswith("s"):
            token = token[:-1]
        tokens.add(token)
    return tokens

def _acronym_supported(obj: str, source_text: str) -> bool:
    generic = {"class", "type", "level", "review", "judgment", "criteria"}
    words = [
        w for w in re.findall(r"[A-Za-z]+", obj)
        if w.lower() not in SOURCE_STOPWORDS | generic
    ]
    initials = "".join(w[0] for w in words).upper()
    source_upper = set(re.findall(r"\b[A-Z][A-Z0-9-]{1,}\b", source_text))
    return any(
        len(candidate) >= 2 and candidate in source_upper
        for candidate in (initials, initials[:2], initials[:3])
    )

def _source_supports_leaf(s: str, p: str, o: str, source_text: str) -> bool:
    if PROMPT_LEAK_RE.search(" ".join((s, p, o))):
        return False
    if re.fullmatch(r"(?:Inclusion|Exclusion) Criterion \d+", o):
        return True
    if not _numeric_object_supported(o, source_text):
        return False
    obj_tokens = _content_tokens(o)
    if len(obj_tokens) < 3:
        return True
    if obj_tokens & _content_tokens(source_text):
        return True
    if _numeric_tokens(o) or _acronym_supported(o, source_text):
        return True
    # Three-or-more unsupported content words are almost always exemplar or
    # prompt leakage (gold audit: <=0.1% after numeric/acronym exceptions).
    return False

def _is_placeholder_triple(s: str, p: str, o: str) -> bool:
    if s in PLACEHOLDERS or p in PLACEHOLDERS or o in PLACEHOLDERS:
        return True
    if PLACEHOLDER_TAIL.search(o) or PLACEHOLDER_TAIL.search(s):
        return True
    if PROMPT_LEAK_RE.search(" ".join((s, p, o))):
        return True
    if any(x.strip() in {"...", "…"} or x.rstrip().endswith("...") for x in (s, p, o)):
        return True
    if all(re.fullmatch(r"[A-F]", x) for x in (s, p, o)):
        return True
    return False

def _normalize_dashes(text: str) -> str:
    """Normalize only separators between quoted fields, never prose dashes."""
    return re.sub(r'"\s*[-–—―]+\s*"', '" — "', text)

def _valid_field_lengths(s: str, p: str, o: str) -> bool:
    # Limits are deliberately above the train maxima (39 / 47 / 244 chars).
    return 0 < len(s) <= 100 and 0 < len(p) <= 100 and 0 < len(o) <= 500

def extract_model_triples(raw: str) -> list[tuple[str, str, str]]:
    """Extract only complete, line-anchored triples from an LLM response.

    The train-label regex intentionally remains permissive.  Applying it to raw
    generation was unsafe: it could connect quoted fragments across many lines
    of chain-of-thought and turn the prose into one giant, technically matching
    triple.  This extractor cannot cross a physical line.
    """
    if not isinstance(raw, str):
        return []
    text = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL)
    text = re.sub(r"```[a-zA-Z]*\n?|```", "", text)
    text = _normalize_dashes(text)
    triples: list[tuple[str, str, str]] = []
    for line in text.splitlines():
        line = re.sub(
            r'^\s*(?:(?:[-*•]|\d+[.)])\s+|\*{0,2}(?:Triple|Detail)\s+\d+\s*:\*{0,2}\s*)',
            "", line, flags=re.IGNORECASE,
        ).strip()
        parts = re.split(r'\s*—\s*', line)
        if len(parts) != 3 or any(len(x) < 2 or not (x.startswith('"') and x.endswith('"')) for x in parts):
            continue
        # The official serializer cannot represent a literal inner quote.
        # Preserve its wording with an apostrophe instead of dropping the row.
        t = tuple(x[1:-1].strip().replace('"', "'") for x in parts)
        if _is_placeholder_triple(*t) or not _valid_field_lengths(*t):
            continue
        triples.append(t)
    return triples

def _has_coherent_model_structure(triples: list[tuple[str, str, str]]) -> bool:
    parents = {
        o for s, p, o in triples
        if s == "Study" and p in BACKBONE_PREDICATES
    }
    linked_parents = {s for s, p, _ in triples if p == "includes criterion"}
    return bool(parents & linked_parents)

def _normalize_regime_parents(
    triple: tuple[str, str, str], regime: str | None,
) -> tuple[str, str, str]:
    if regime not in {"set", "plain"}:
        return triple
    s, p, o = triple
    wanted = {
        "Inclusion Criteria": "Inclusion Criteria Set" if regime == "set" else "Inclusion Criteria",
        "Inclusion Criteria Set": "Inclusion Criteria Set" if regime == "set" else "Inclusion Criteria",
        "Exclusion Criteria": "Exclusion Criteria Set" if regime == "set" else "Exclusion Criteria",
        "Exclusion Criteria Set": "Exclusion Criteria Set" if regime == "set" else "Exclusion Criteria",
    }
    return wanted.get(s, s), p, wanted.get(o, o)

def _drop_out_of_range_orphan_links(
    triples: list[tuple[str, str, str]], source_text: str,
) -> list[tuple[str, str, str]]:
    """Drop only structurally impossible, leafless Criterion-N links.

    An early tail smoke run emitted an Inclusion Criterion 5 link for a source
    with four top-level inclusion items, but emitted no facts for that node.
    Such orphans occur in only 1/1,116 train ``includes criterion`` links.  A
    link is retained whenever its node has a detail triple, even if the list
    parser under-counts a genuinely nested source structure.
    """
    inc_text, exc_text = split_sections(source_text)
    limits = {
        "Inclusion": len(split_items(inc_text)),
        "Exclusion": len(split_items(exc_text)),
    }
    leaf_subjects = {
        s for s, p, _ in triples if p not in STRUCTURAL_PREDICATES
    }
    criterion_re = re.compile(r"^(Inclusion|Exclusion) Criterion (\d+)$")
    kept = []
    for t in triples:
        _, p, o = t
        match = criterion_re.fullmatch(o) if p == "includes criterion" else None
        if match:
            kind, raw_n = match.groups()
            if int(raw_n) > limits[kind] and o not in leaf_subjects:
                continue
        kept.append(t)
    return kept

def clean_output(
    raw: str, *, skeleton_first: list[tuple[str, str, str]] | None = None,
    source_text: str | None = None, annotation_regime: str | None = None,
) -> str:
    model_triples = [
        _normalize_regime_parents(t, annotation_regime)
        for t in extract_model_triples(raw)
    ]

    # Prefer a coherent structure produced by the model.  The old pipeline
    # always prepended its heuristic skeleton, but that skeleton is exact on
    # only 65/100 train rows (nested lists and multi-cohort studies are the main
    # failures).  It remains a useful fallback when generation omits structure.
    seed_triples = (
        model_triples
        if _has_coherent_model_structure(model_triples)
        else [
            *(_normalize_regime_parents(t, annotation_regime) for t in (skeleton_first or [])),
            *model_triples,
        ]
    )
    if source_text:
        seed_triples = _drop_out_of_range_orphan_links(seed_triples, source_text)

    # Preserve order, dedupe, drop placeholders, and keep only the first object
    # for each Study/backbone relation.
    #    drops placeholder rows, keeps only the first object for each backbone
    #    (Subject, Predicate) pair so the model can't double-emit "Study →
    #    has inclusion criteria → ..." with two different objects).
    seen_triples: set[tuple[str, str, str]] = set()
    seen_backbone: set[tuple[str, str]] = set()
    triples: list[tuple[str, str, str]] = []

    def add(s: str, p: str, o: str) -> None:
        if _is_placeholder_triple(s, p, o):
            return
        # Qwen occasionally emits redundant metadata such as
        # ``stroke synonym -> CVA`` after already extracting the clinical fact.
        # No train predicate contains "synonym"; keeping it only adds a likely
        # unmatched row under the Hungarian metric.
        if REDUNDANT_META_PREDICATE_RE.search(p):
            return
        if source_text and p not in STRUCTURAL_PREDICATES and not _source_supports_leaf(s, p, o, source_text):
            return
        t = (s, p, o)
        if t in seen_triples:
            return
        if p in BACKBONE_PREDICATES and (s, p) in seen_backbone:
            return
        seen_triples.add(t); seen_backbone.add((s, p)); triples.append(t)

    for s, p, o in seed_triples:
        add(s, p, o)

    return TRIPLE_DELIM.join(f'"{s}"{SEP}"{p}"{SEP}"{o}"' for s, p, o in triples)

def repair_prediction(
    text: str, *, source_text: str | None = None,
    annotation_regime: str | None = None,
) -> str:
    """Re-apply the same cleanup to an existing cleaned prediction. Useful for
    sweeping older checkpoints after improving clean_output."""
    return clean_output(
        text, skeleton_first=None, source_text=source_text,
        annotation_regime=annotation_regime,
    )

def _structural_coverage(prediction: str, source_text: str) -> float:
    triples = parse_triples(prediction)
    emitted = {o for _, p, o in triples if p == "includes criterion"}
    inc_text, exc_text = split_sections(source_text)
    required = {
        *(f"Inclusion Criterion {i}" for i in range(1, len(split_items(inc_text)) + 1)),
        *(f"Exclusion Criterion {i}" for i in range(1, len(split_items(exc_text)) + 1)),
    }
    return len(emitted & required) / len(required) if required else 1.0

def _should_review(first: str, source_text: str, meta: dict) -> bool:
    if REVIEW_MODE == "off":
        return False
    if REVIEW_MODE == "all":
        return True
    triples = parse_triples(first)
    center, lower, upper = expected_triple_count(source_text, meta["annotation_regime"])
    return (
        meta.get("finish_reason") == "length"
        or len(triples) < lower
        or len(triples) > upper
        or _structural_coverage(first, source_text) < 0.8
    )

def choose_reviewed_candidate(
    first: str, reviewed: str, source_text: str,
) -> tuple[str, str]:
    """Accept a review unless objective completeness safeguards regress."""
    first_ts, reviewed_ts = parse_triples(first), parse_triples(reviewed)
    if not _has_coherent_model_structure(reviewed_ts):
        return first, "rejected_incoherent"
    if len(reviewed_ts) < max(4, int(0.70 * len(first_ts))):
        return first, "rejected_too_short"
    _, _, upper = expected_triple_count(source_text)
    if len(reviewed_ts) > 1.5 * max(1, len(first_ts)) and len(reviewed_ts) > upper + 10:
        return first, "rejected_bloated"
    first_coverage = _structural_coverage(first, source_text)
    reviewed_coverage = _structural_coverage(reviewed, source_text)
    if reviewed_coverage + 0.10 < first_coverage:
        return first, "rejected_coverage"
    return reviewed, "accepted"

## 10. Smoke test on held-out train rows (no LLM call yet)

Verify that the skeleton + retrieval + formatting all work end-to-end on a
couple of train rows whose ground truth we can compare against.

In [11]:
def show_diff(pred_triples: list[tuple[str, str, str]], gold_triples: list[tuple[str, str, str]]) -> None:
    g = set(gold_triples); p = set(pred_triples)
    print(f"  predicted={len(p)} | gold={len(g)} | intersection={len(p & g)}")
    miss = list(g - p)[:5]; extra = list(p - g)[:5]
    if miss: print("  e.g. missing:", miss[0])
    if extra: print("  e.g. extra :", extra[0])

# Skeleton-only baseline (no LLM) — sanity check
for i in [0, 1, 2]:
    sk, _, _ = make_skeleton(train_df.iloc[i]["eligibility_criteria"])
    print(f"row {i}:")
    show_diff(sk, train_df.iloc[i]["triples"])

row 0:
  predicted=15 | gold=47 | intersection=15
  e.g. missing: ('Inclusion Criterion 3', 'timing', 'prior to the acute event')
row 1:
  predicted=26 | gold=50 | intersection=0
  e.g. missing: ('Exclusion Criterion 2', 'life expectancy', 'less than 6 months')
  e.g. extra : ('Study', 'has exclusion criteria', 'Exclusion Criteria')
row 2:
  predicted=4 | gold=12 | intersection=0
  e.g. missing: ('Inclusion Criterion 1', 'smoking requirement', 'nonsmokers')
  e.g. extra : ('Study', 'has inclusion criteria', 'Inclusion Criteria')


## 11. Full inference loop over the test sheet

Rows 0-44 are seeded from the scored CSV. Only rows 45-49 enter the no-think
inference loop, with dynamic 1024-4096 token budgets and review disabled.

In [12]:
CHECKPOINT = Path(os.environ.get(
    "COHORTX_CHECKPOINT",
    ROOT / "predictions_v4_tail.jsonl" if ON_KAGGLE else CACHE_DIR / "predictions_v4_tail.jsonl",
))
PROGRESS_CSV = ROOT / "submission_progress.csv"

def _run_signature(k_shots: int) -> str:
    payload = json.dumps({
        "model": MODEL_PATH,
        "model_size": Path(MODEL_PATH).stat().st_size if Path(MODEL_PATH).exists() else None,
        "prompt": SYSTEM_PROMPT,
        "generation": GEN_KWARGS,
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "min_output_tokens": MIN_OUTPUT_TOKENS,
        "review_mode": REVIEW_MODE,
        "k_shots": k_shots,
        "pipeline": "v4-preserve-081-core-generate-tail",
        "base_submission_sha256": BASE_SHA256,
        "tail_exemplars": TAIL_EXEMPLARS,
        "postprocess": 5,
    }, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]

def already_done(run_signature: str) -> dict[int, dict]:
    """Seed rows 0-44 from the scored CSV and resume generated tail rows."""
    done: dict[int, dict] = {
        i: {"i": i, "structured": BASE_PREDICTIONS[i], "base_081": True}
        for i in BASE_KEEP_ROWS
    }
    print("[tail run] preserved 45 scored rows; 5 LLM tail rows required")

    if CHECKPOINT.exists():
        for ln in CHECKPOINT.read_text().splitlines():
            if not ln.strip(): continue
            try:
                d = json.loads(ln)
                if d.get("run_signature") == run_signature:
                    done[d["i"]] = d
            except Exception: pass
    return done

def _complete_with_skeletons(preds: dict[int, str]) -> dict[int, str]:
    """Return a complete, always-submittable snapshot for timeout recovery."""
    complete = dict(preds)
    for i in range(len(test_df)):
        if not complete.get(i):
            text = test_df.iloc[i]["eligibility_criteria"]
            regime = predict_annotation_regime(text)
            suffix = " Set" if regime == "set" else ""
            skeleton, _, _ = make_skeleton(
                text,
                inc_label=f"Inclusion Criteria{suffix}",
                exc_label=f"Exclusion Criteria{suffix}",
            )
            complete[i] = format_skeleton(skeleton)
    return complete

def predict_all(llm=None, limit: int | None = None, k_shots: int = K_SHOTS) -> dict[int, str]:
    """Run inference over the test rows. Pass limit=N for a smoke run."""
    if llm is None: llm = load_llm()
    signature = _run_signature(k_shots)
    done = already_done(signature)
    preds: dict[int, str] = {i: d["structured"] for i, d in done.items()}

    targets = list(range(len(test_df)))[:limit] if limit else list(range(len(test_df)))
    t0 = time.time()
    for n, i in enumerate(targets):
        if i in done:
            print(f"[{n+1}/{len(targets)}] row {i}: cached")
            continue
        wall_elapsed = time.time() - PROCESS_START
        if ON_KAGGLE and wall_elapsed >= MAX_WALL_SECONDS:
            remaining = [j for j in targets[n:] if j not in preds]
            print(
                f"[time guard] {wall_elapsed:.0f}s elapsed; using skeleton fallback "
                f"for {len(remaining)} remaining rows so submission.csv is written "
                "before Kaggle's 43200s limit."
            )
            return _complete_with_skeletons(preds)
        text = test_df.iloc[i]["eligibility_criteria"]
        regime = predict_annotation_regime(text)
        suffix = " Set" if regime == "set" else ""
        skeleton, _, _ = make_skeleton(
            text,
            inc_label=f"Inclusion Criteria{suffix}",
            exc_label=f"Exclusion Criteria{suffix}",
        )
        raw, meta = llm_generate(llm, text, k=k_shots, test_idx=i)
        cleaned = clean_output(
            raw, skeleton_first=skeleton, source_text=text,
            annotation_regime=regime,
        )
        first_cleaned = cleaned
        review_raw = ""
        review_meta: dict = {"review_decision": "not_run"}
        review_allowed = not ON_KAGGLE or (time.time() - PROCESS_START) < MAX_WALL_SECONDS
        wants_review = _should_review(first_cleaned, text, meta)
        if wants_review and review_allowed:
            review_raw, call_meta = llm_review(
                llm, text, first_cleaned, regime=regime,
            )
            reviewed = clean_output(
                review_raw, skeleton_first=skeleton, source_text=text,
                annotation_regime=regime,
            )
            cleaned, decision = choose_reviewed_candidate(
                first_cleaned, reviewed, text,
            )
            review_meta = {**call_meta, "review_decision": decision}
        elif wants_review and not review_allowed:
            review_meta = {"review_decision": "skipped_time_guard"}
        preds[i] = cleaned
        with CHECKPOINT.open("a") as f:
            f.write(json.dumps({
                "i": i,
                "run_signature": signature,
                "structured": cleaned,
                "first_structured": first_cleaned,
                "raw": raw,
                "review_raw": review_raw,
                **meta,
                **review_meta,
            }, ensure_ascii=False) + "\n")
        n_triples = cleaned.count(SEP) // 2
        print(
            f"[{n+1}/{len(targets)}] row {i}: {n_triples} triples | "
            f"{meta['annotation_regime']} | {meta['finish_reason']} | "
            f"review={review_meta['review_decision']} | "
            f"{time.time()-t0:.0f}s total"
        )
        # Visible, valid recovery artifact after every completed row. If Kaggle
        # kills the current generation, the previous snapshot is still usable.
        write_submission(_complete_with_skeletons(preds), PROGRESS_CSV, quiet=True)
    return preds

# To run end-to-end, uncomment:
# llm = load_llm()
# preds = predict_all(llm)

## 12. Build and write the submission CSV

Same layout as the train sheet — two columns, no extra index.
Newlines and quotes inside `structured` are escaped per RFC 4180 (pandas's
default), which Kaggle's reader accepts.

In [13]:
def validate_prediction(text: str) -> list[str]:
    """Return format errors for one canonical `structured` cell."""
    errors = []
    if not text:
        return ["empty prediction"]
    segments = text.split(TRIPLE_DELIM)
    for j, segment in enumerate(segments):
        if not STRICT_TRIPLE_RE.fullmatch(segment):
            errors.append(f"segment {j} is not one strict triple")
    if len(parse_triples(text)) != len(segments):
        errors.append("parser count differs from blank-line segment count")
    return errors

def write_submission(
    preds: dict[int, str], out_path: Path = SUB_CSV, *, allow_partial: bool = False,
    quiet: bool = False,
) -> None:
    rows = []
    for i in range(len(test_df)):
        pred_text = preds.get(i, "")
        source_text = test_df.iloc[i]["eligibility_criteria"]
        if i in BASE_KEEP_ROWS:
            # Preserve the exact leaderboard-scored cell. Validate it, but do
            # not normalize, dedupe, ground-filter or otherwise mutate it.
            pred_text = BASE_PREDICTIONS[i]
        elif pred_text:
            pred_text = repair_prediction(
                pred_text,
                source_text=source_text,
                annotation_regime=predict_annotation_regime(source_text),
            )
        errors = validate_prediction(pred_text)
        if errors and not allow_partial:
            raise ValueError(f"row {i}: {'; '.join(errors)}")
        rows.append({
            "eligibility_criteria": test_df.iloc[i]["eligibility_criteria"],
            "structured": pred_text,
        })
    pd.DataFrame(rows).to_csv(out_path, index=False, quoting=csv.QUOTE_ALL,
                              encoding="utf-8", lineterminator="\n")
    # Verify the exact bytes we wrote round-trip through the standard CSV
    # parser and preserve test-row order.
    with out_path.open(newline="", encoding="utf-8") as f:
        roundtrip = list(csv.DictReader(f))
    if len(roundtrip) != len(test_df):
        raise ValueError(f"submission has {len(roundtrip)} rows, expected {len(test_df)}")
    for i, row in enumerate(roundtrip):
        if row["eligibility_criteria"] != test_df.iloc[i]["eligibility_criteria"]:
            raise ValueError(f"row {i}: eligibility_criteria changed during CSV round-trip")
        if i in BASE_KEEP_ROWS and row["structured"] != BASE_PREDICTIONS[i]:
            raise ValueError(f"row {i}: scored 0.81 structured cell changed")
        errors = validate_prediction(row["structured"])
        if errors and not allow_partial:
            raise ValueError(f"round-trip row {i}: {'; '.join(errors)}")
    if not quiet:
        print(f"wrote {out_path} ({out_path.stat().st_size} bytes)")

# Run when predictions exist:
# write_submission(preds)

## 13. Entry point — run end-to-end and write the submission

Kaggle's **Run All** imports rows 0-44 from one attached
`submission_081.csv`, predicts rows 45-49 with review disabled,
validates every triple and CSV round-trip, then writes
`/kaggle/working/submission.csv`.

Paths are discovered automatically; override with env vars when needed:

```bash
# Local Mac (default)
python kaggle_submission_v4_tail.py

# Kaggle notebook (after uploading a GGUF model as a dataset)
COHORTX_MODEL=/kaggle/input/qwen35-9b-gguf/Qwen3.5-9B-Q8_0.gguf python kaggle_submission_v4_tail.py

# Anywhere with custom paths
COHORTX_XLSX=./Task_2.xlsx COHORTX_BASE_SUBMISSION=./submission_081.csv python kaggle_submission_v4_tail.py
```

Runtime depends strongly on Kaggle's assigned CPU. Per-row JSONL checkpoints
make interruption/resume safe.

In [14]:
if __name__ == "__main__":
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--limit", type=int, default=None,
                    help="only run N test rows (smoke / partial)")
    ap.add_argument("--skip-llm", action="store_true",
                    help="skip LLM, fill predictions with skeleton-only output")
    ap.add_argument("--out", type=str, default=str(SUB_CSV),
                    help="output submission CSV path")
    ap.add_argument("--allow-partial", action="store_true",
                    help="allow empty rows in a smoke-test CSV (never submit it)")
    args, _ = ap.parse_known_args()

    if args.skip_llm:
        preds = {}
        for i in range(len(test_df)):
            text = test_df.iloc[i]["eligibility_criteria"]
            regime = predict_annotation_regime(text)
            suffix = " Set" if regime == "set" else ""
            skel, _, _ = make_skeleton(
                text,
                inc_label=f"Inclusion Criteria{suffix}",
                exc_label=f"Exclusion Criteria{suffix}",
            )
            preds[i] = format_skeleton(skel)
        print(f"skeleton-only predictions for {len(preds)} rows")
    else:
        llm = load_llm()
        preds = predict_all(llm, limit=args.limit)

    write_submission(preds, Path(args.out), allow_partial=args.allow_partial)

[tail run] preserved 45 scored rows; 5 LLM tail rows required
[1/50] row 0: cached
[2/50] row 1: cached
[3/50] row 2: cached
[4/50] row 3: cached
[5/50] row 4: cached
[6/50] row 5: cached
[7/50] row 6: cached
[8/50] row 7: cached
[9/50] row 8: cached
[10/50] row 9: cached
[11/50] row 10: cached
[12/50] row 11: cached
[13/50] row 12: cached
[14/50] row 13: cached
[15/50] row 14: cached
[16/50] row 15: cached
[17/50] row 16: cached
[18/50] row 17: cached
[19/50] row 18: cached
[20/50] row 19: cached
[21/50] row 20: cached
[22/50] row 21: cached
[23/50] row 22: cached
[24/50] row 23: cached
[25/50] row 24: cached
[26/50] row 25: cached
[27/50] row 26: cached
[28/50] row 27: cached
[29/50] row 28: cached
[30/50] row 29: cached
[31/50] row 30: cached
[32/50] row 31: cached
[33/50] row 32: cached
[34/50] row 33: cached
[35/50] row 34: cached
[36/50] row 35: cached
[37/50] row 36: cached
[38/50] row 37: cached
[39/50] row 38: cached
[40/50] row 39: cached
[41/50] row 40: cached
[42/50] row 41